In [ ]:
!pip install -q pika

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 6.0 MB/s eta 0:00:00


In [ ]:
import pika
import json
import os
from getpass import getpass

RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"

RABBITMQ_PASSWORD = getpass("Enter RabbitMQ password: ")

print("RabbitMQ configuration loaded successfully!")
print("Host:", RABBITMQ_HOST)
print("Port:", RABBITMQ_PORT)
print("Username:", RABBITMQ_USERNAME)

Enter RabbitMQ password: ··········
RabbitMQ configuration loaded successfully!
Host: 129.153.75.221
Port: 5672
Username: bytesmart_interns


In [ ]:
import pika

credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

connection_params = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    credentials=credentials
)

try:
    connection = pika.BlockingConnection(connection_params)
    channel = connection.channel()

    print("RabbitMQ connection successful!")
    print("Channel created successfully!")

except Exception as e:
    print("RabbitMQ connection failed!")
    print("Error:", e)

RabbitMQ connection successful!
Channel created successfully!


In [ ]:
QUEUE_NAME = "faculty_burnout_queue"

channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print("Queue created successfully!")
print("Queue name:", QUEUE_NAME)

Queue created successfully!
Queue name: faculty_burnout_queue


In [ ]:
message = {
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 35,
    "Advising_Students": 25,
    "Committee_Count": 6,
    "Research_Hours": 5,
    "Admin_Hours": 12,
    "Semester_Progress": 0.9,
    "Historical_Leave_Days": 10,
    "Burnout_Risk": "High"
}

message_body = json.dumps(message)

channel.basic_publish(
    exchange="",
    routing_key=QUEUE_NAME,
    body=message_body,
    properties=pika.BasicProperties(
        delivery_mode=2
    )
)

print("Message published successfully!")
print("Queue:", QUEUE_NAME)
print("Message:", message_body)

Message published successfully!
Queue: faculty_burnout_queue
Message: {"Faculty_ID": "FAC001", "Teaching_Hours": 35, "Advising_Students": 25, "Committee_Count": 6, "Research_Hours": 5, "Admin_Hours": 12, "Semester_Progress": 0.9, "Historical_Leave_Days": 10, "Burnout_Risk": "High"}


In [ ]:
method_frame, header_frame, body = channel.basic_get(
    queue=QUEUE_NAME,
    auto_ack=False
)

if method_frame:
    received_message = json.loads(body.decode("utf-8"))

    print("Message received successfully!")
    print()
    print("Received Message:")
    print(received_message)

    channel.basic_ack(
        delivery_tag=method_frame.delivery_tag
    )

    print()
    print("Message acknowledged successfully!")

else:
    print("No message available in the queue.")

Message received successfully!

Received Message:
{'Faculty_ID': 'FAC001', 'Teaching_Hours': 35, 'Advising_Students': 25, 'Committee_Count': 6, 'Research_Hours': 5, 'Admin_Hours': 12, 'Semester_Progress': 0.9, 'Historical_Leave_Days': 10, 'Burnout_Risk': 'High'}

Message acknowledged successfully!


In [ ]:
if "received_message" in globals():

    faculty_id = received_message["Faculty_ID"]

    teaching = received_message["Teaching_Hours"]
    advising = received_message["Advising_Students"]
    committee = received_message["Committee_Count"]
    research = received_message["Research_Hours"]
    admin = received_message["Admin_Hours"]
    semester = received_message["Semester_Progress"]
    leave_days = received_message["Historical_Leave_Days"]
    burnout_risk = received_message["Burnout_Risk"]

    total_responsibilities = (
        teaching +
        advising +
        committee +
        research +
        admin
    )

    processed_result = {
        "Faculty_ID": faculty_id,
        "Burnout_Risk": burnout_risk,
        "Total_Responsibility_Value": total_responsibilities,
        "Semester_Progress": semester,
        "Historical_Leave_Days": leave_days,
        "Processing_Status": "Processed Successfully"
    }

    print("Message processed successfully!")
    print()
    print("Processed Result:")
    print(processed_result)

else:
    print("No received message found.")

Message processed successfully!

Processed Result:
{'Faculty_ID': 'FAC001', 'Burnout_Risk': 'High', 'Total_Responsibility_Value': 83, 'Semester_Progress': 0.9, 'Historical_Leave_Days': 10, 'Processing_Status': 'Processed Successfully'}


In [ ]:
import pandas as pd
import json
import joblib
import os

# Load the trained ML model
model_path = "/content/faculty_burnout_model_pipeline.pkl"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        "Model file not found. Please run Notebook 2 or upload the .pkl file."
    )

model = joblib.load(model_path)

print("ML model loaded successfully!")

# Faculty data
faculty_data = {
    "Teaching_Hours": 35,
    "Advising_Students": 25,
    "Committee_Count": 6,
    "Research_Hours": 5,
    "Admin_Hours": 12,
    "Semester_Progress": 0.9,
    "Historical_Leave_Days": 10
}

# Convert to DataFrame
input_data = pd.DataFrame([faculty_data])

# Generate prediction
prediction = model.predict(input_data)[0]

# Generate prediction probabilities
probabilities = model.predict_proba(input_data)[0]
class_names = model.classes_

risk_probabilities = {
    str(class_name): round(float(probability), 4)
    for class_name, probability in zip(class_names, probabilities)
}

# Create RabbitMQ message
ml_message = {
    "Faculty_ID": "FAC001",
    **faculty_data,
    "Burnout_Risk": str(prediction),
    "Risk_Probabilities": risk_probabilities
}

# Convert message to JSON
ml_message_body = json.dumps(ml_message)

print()
print("ML prediction generated successfully!")
print()
print("Prediction:", prediction)
print("Risk Probabilities:", risk_probabilities)
print()
print("RabbitMQ Message:")
print(ml_message_body)

ML model loaded successfully!

ML prediction generated successfully!

Prediction: High
Risk Probabilities: {'High': 0.6028, 'Low': 0.0067, 'Medium': 0.3906}

RabbitMQ Message:
{"Faculty_ID": "FAC001", "Teaching_Hours": 35, "Advising_Students": 25, "Committee_Count": 6, "Research_Hours": 5, "Admin_Hours": 12, "Semester_Progress": 0.9, "Historical_Leave_Days": 10, "Burnout_Risk": "High", "Risk_Probabilities": {"High": 0.6028, "Low": 0.0067, "Medium": 0.3906}}


In [ ]:
import pika

print("Reconnecting to RabbitMQ...")

credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

connection_params = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    credentials=credentials,
    heartbeat=60,
    blocked_connection_timeout=30
)

try:
    connection = pika.BlockingConnection(connection_params)
    channel = connection.channel()

    # Make sure the queue exists
    channel.queue_declare(
        queue=QUEUE_NAME,
        durable=True
    )

    print("RabbitMQ reconnected successfully!")
    print("Channel recreated successfully!")
    print("Queue:", QUEUE_NAME)

except Exception as e:
    print("RabbitMQ reconnection failed!")
    print("Error:", e)

Reconnecting to RabbitMQ...
RabbitMQ reconnected successfully!
Channel recreated successfully!
Queue: faculty_burnout_queue


In [ ]:
try:
    channel.basic_publish(
        exchange="",
        routing_key=QUEUE_NAME,
        body=ml_message_body,
        properties=pika.BasicProperties(
            delivery_mode=2
        )
    )

    print("ML prediction message published successfully!")
    print()
    print("Queue:", QUEUE_NAME)
    print("Message:")
    print(ml_message_body)

except Exception as e:
    print("Message publishing failed!")
    print("Error:", e)

ML prediction message published successfully!

Queue: faculty_burnout_queue
Message:
{"Faculty_ID": "FAC001", "Teaching_Hours": 35, "Advising_Students": 25, "Committee_Count": 6, "Research_Hours": 5, "Admin_Hours": 12, "Semester_Progress": 0.9, "Historical_Leave_Days": 10, "Burnout_Risk": "High", "Risk_Probabilities": {"High": 0.6028, "Low": 0.0067, "Medium": 0.3906}}


In [ ]:
try:
    method_frame, header_frame, body = channel.basic_get(
        queue=QUEUE_NAME,
        auto_ack=False
    )

    if method_frame:
        received_ml_message = json.loads(body.decode("utf-8"))

        print("ML message received successfully!")
        print()
        print("Received Message:")
        print(json.dumps(received_ml_message, indent=4))

        # Acknowledge the message
        channel.basic_ack(
            delivery_tag=method_frame.delivery_tag
        )

        print()
        print("Message acknowledged successfully!")

    else:
        print("No message available in the queue.")

except Exception as e:
    print("Message consumption failed!")
    print("Error:", e)

ML message received successfully!

Received Message:
{
    "Faculty_ID": "FAC001",
    "Teaching_Hours": 35,
    "Advising_Students": 25,
    "Committee_Count": 6,
    "Research_Hours": 5,
    "Admin_Hours": 12,
    "Semester_Progress": 0.9,
    "Historical_Leave_Days": 10,
    "Burnout_Risk": "High",
    "Risk_Probabilities": {
        "High": 0.6028,
        "Low": 0.0067,
        "Medium": 0.3906
    }
}

Message acknowledged successfully!


In [ ]:
if "received_ml_message" in globals():

    faculty_id = received_ml_message["Faculty_ID"]
    burnout_risk = received_ml_message["Burnout_Risk"]
    risk_probabilities = received_ml_message["Risk_Probabilities"]

    print("===== Faculty Burnout Message Processing =====")
    print()
    print("Faculty ID:", faculty_id)
    print("Burnout Risk:", burnout_risk)
    print("Risk Probabilities:", risk_probabilities)

    # Simple processing logic
    if burnout_risk == "High":
        action = "Faculty workload should be reviewed."
    elif burnout_risk == "Medium":
        action = "Faculty workload should be monitored."
    else:
        action = "Faculty workload is currently within a lower-risk range."

    processed_result = {
        "Faculty_ID": faculty_id,
        "Burnout_Risk": burnout_risk,
        "Risk_Probabilities": risk_probabilities,
        "Recommended_Action": action,
        "Processing_Status": "Completed"
    }

    print()
    print("Processed Result:")
    print(json.dumps(processed_result, indent=4))

else:
    print("No ML message found for processing.")

===== Faculty Burnout Message Processing =====

Faculty ID: FAC001
Burnout Risk: High
Risk Probabilities: {'High': 0.6028, 'Low': 0.0067, 'Medium': 0.3906}

Processed Result:
{
    "Faculty_ID": "FAC001",
    "Burnout_Risk": "High",
    "Risk_Probabilities": {
        "High": 0.6028,
        "Low": 0.0067,
        "Medium": 0.3906
    },
    "Recommended_Action": "Faculty workload should be reviewed.",
    "Processing_Status": "Completed"
}


In [ ]:
print()
print("   PHASE 3 - RABBITMQ FLOW VERIFICATION")
print()

print()
print("1. RabbitMQ Connection : SUCCESS")
print("2. Queue Created       :", QUEUE_NAME)
print("3. ML Prediction       :", received_ml_message["Burnout_Risk"])
print("4. Message Published   : SUCCESS")
print("5. Message Consumed    : SUCCESS")
print("6. Message Processed   :", processed_result["Processing_Status"])

print()
print()
print("       PHASE 3 COMPLETED SUCCESSFULLY")
print()

print()
print("Flow:")
print("ML Model")
print("   ↓")
print("Producer")
print("   ↓")
print("RabbitMQ Queue")
print("   ↓")
print("Consumer")
print("   ↓")
print("Message Processing")


   PHASE 3 - RABBITMQ FLOW VERIFICATION


1. RabbitMQ Connection : SUCCESS
2. Queue Created       : faculty_burnout_queue
3. ML Prediction       : High
4. Message Published   : SUCCESS
5. Message Consumed    : SUCCESS
6. Message Processed   : Completed


       PHASE 3 COMPLETED SUCCESSFULLY


Flow:
ML Model
   ↓
Producer
   ↓
RabbitMQ Queue
   ↓
Consumer
   ↓
Message Processing


In [ ]:

print("       PHASE 3 - RABBITMQ CONFIGURATION")


print()
print("Messaging Framework : Pika (Python RabbitMQ Client)")
print("RabbitMQ Host       :", RABBITMQ_HOST)
print("RabbitMQ Port       :", RABBITMQ_PORT)
print("RabbitMQ Username   :", RABBITMQ_USERNAME)
print("RabbitMQ Queue      :", QUEUE_NAME)

print()

print("MESSAGE FLOW:")

print()

print("1. ML model generates faculty burnout prediction")
print("2. Producer creates JSON message")
print("3. Producer publishes message to RabbitMQ")
print("4. RabbitMQ stores message in the queue")
print("5. Consumer receives the message")
print("6. Consumer acknowledges the message")
print("7. Consumer processes the received prediction")

print()

print("PHASE 3 STATUS:")

print()
print("Connection        : SUCCESS")
print("Queue             : CREATED")
print("Message Publishing: SUCCESS")
print("Message Consuming  : SUCCESS")
print("Message Processing : SUCCESS")


       PHASE 3 - RABBITMQ CONFIGURATION

Messaging Framework : Pika (Python RabbitMQ Client)
RabbitMQ Host       : 129.153.75.221
RabbitMQ Port       : 5672
RabbitMQ Username   : bytesmart_interns
RabbitMQ Queue      : faculty_burnout_queue

MESSAGE FLOW:

1. ML model generates faculty burnout prediction
2. Producer creates JSON message
3. Producer publishes message to RabbitMQ
4. RabbitMQ stores message in the queue
5. Consumer receives the message
6. Consumer acknowledges the message
7. Consumer processes the received prediction

PHASE 3 STATUS:

Connection        : SUCCESS
Queue             : CREATED
Message Publishing: SUCCESS
Message Consuming  : SUCCESS
Message Processing : SUCCESS
